# `RUN__pdf_ocr` — the CafeF filing parse, in two modes

Runs the production OCR cascade over one ticker's filings and writes a **run folder**, never a
statement CSV. Two modes, **the same code**, and `MODE` says which one out loud:

| `MODE` | reads | runs on |
|---|---|---|
| `"local"` | `raw_data/cafef/` in this repo | this machine's card (RTX 3050, 4 GiB) |
| `"kgpu"` | the payload `kgpu_bootstrap` mounted | a Kaggle T4 (15 GiB) |
| `"auto"` | whichever is present | — the default, and it PRINTS what it resolved to |

    cd src\\kaggle_gpu
    python -m kgpu rehearse pdf-ocr     # the worker side, locally, no quota
    python -m kgpu run     pdf-ocr      # push, wait, download, merge

⚠️ **`auto` NEVER GUESSES SILENTLY.** It resolves from `$CAFEF_DATA_ROOT` — the variable the
bootstrap sets — and prints the mode and the root it chose. An explicit `MODE` is ASSERTED: ask
for `"kgpu"` where no payload is mounted and the notebook raises rather than quietly parsing
the repo's own `raw_data/`, which would look like a successful Kaggle run on the wrong machine.

## ⚠️ What it does NOT do

1. **It writes no statement CSV.** Merging a recovered quarter back into
   `raw_data/cafef/financials/statements/` stays a deliberate act through Dagster, with a
   pre-run backup and a diff of every column — this repo has measured four runs in which a
   `periods` build silently DOWNGRADED a quarter it was given only for history, while the log
   said `RUN_SUCCESS` (CLAUDE.md §6-2-vicies, §6-2-unvicies, §6-2-quatervicies,
   §6-2-quinvicies).
2. **It does not retry an ALTERNATE filing** and **does not de-cumulate**. Both are `build()`'s
   and both need state a one-document run has not got, so `compare()` REFUSES to score a
   cumulative income statement rather than reporting every cell as changed.
3. ⚠️ **It does not make a non-bank template SAFE.** `TPL-1`: seven reconcile anchors are
   bank-shaped, and on `corp` and `insurance` the cash-flow one fuzzy-matches the OPENING
   balance and returns it as the closing one — a wrong figure, not a refusal.

## What the log's percentages mean — three denominators, one predicts time

| line | denominator | predicts time? |
|---|---|---|
| `[doc 2/3   67% of DOCUMENTS]` | documents | ❌ 4.2 min against 18.2 for a failing one |
| `[layer 12/47  26% of POSITIONS]` | positions in the cascade | ❌ one layer re-OCRs every page, the next re-maps a cache in ms |
| `[ocr page 40/96  42% of PAGES]` | pages of one OCR pass | ✅ **the only one** — ~0.87 s/page |

In [ ]:
# ── PARAMETERS ────────────────────────────────────────────────────────────────
# ⚠️ Rewritten IN PLACE by `kgpu build` (src/kaggle_gpu/kaggle_config.json), so every name
# below must stay a TOP-LEVEL assignment. A parameter kgpu cannot find refuses the push.
MODE = "auto"                  # "auto" | "local" | "kgpu" — auto prints what it resolved to
EXCHANGE = "HOSE"
SYMBOL = "VCB"
PERIODS = ["Q1-2026"]          # None = every period this ticker files. A period it does not file RAISES.
ALLOW_PARENT = False           # fall back to the STANDALONE filing where no consolidated one exists
TEMPLATE = None                # None = RESOLVE (templates.csv, then CafeF's fingerprint). ⚠️ Never defaults to "bank".
LAYERS = None                  # None = the full 47-layer cascade; else a list of layer names, in cascade order
COMPARE = True                 # score every parsed cell against the statement CSV on disk
OUT_ROOT = None                # None = <repo>/reports/pdf_ocr, anchored to the module file rather than the CWD
NOTES = ""

# ⚠️ **NOT A KNOB: the engine.** Every `ParseLayer` names its own engine and DPI, so the
# cascade decides what runs — `onnx` for 43 of the 47 layers, `tesseract` for the four that
# are skipped wherever it is not installed (a Kaggle worker, for one).


In [ ]:
# ── DEPENDENCIES — ONE PINNED FILE, BOTH MACHINES ─────────────────────────────
# ⚠️ **`src/web_scraper/requirements-ocr.txt` IS THE SINGLE SOURCE OF TRUTH**, installed here
# and by hand into `mt_env`. Every line in it changes pixels, boxes or characters: the
# rasteriser, the DB detector, the recogniser, the resize/normalise, the polygon unclip and the
# arrays between them.
#
# ⚠️ **EVERY REQUIREMENT IS HANDED TO pip, NOT JUST THE ONES THAT LOOK UNSATISFIED.** The first
# version of this cell reconstructed `==` pins from the file and installed only those — so the
# day `onnxruntime-gpu` became a RANGE it was silently never installed, and the run died at the
# import check blaming a missing internet connection it had. pip is a no-op on an already
# satisfied requirement; the reconstruction was the bug.
#
# ⚠️ `vietocr` gets `--no-deps`: its metadata pins `pillow==10.2.0` and `albumentations==1.4.2`,
# which pip would downgrade a whole image to satisfy, and the predictor path needs neither.
import importlib.util
import subprocess
import sys
from pathlib import Path

_REQ = None
for _base in (Path.cwd(), *Path.cwd().parents):
    _c = _base / "src" / "web_scraper" / "requirements-ocr.txt"
    if _c.is_file():
        _REQ = _c
        break
if _REQ is None:
    raise FileNotFoundError(
        "requirements-ocr.txt not found. On a worker it travels in source.zip, which "
        "kgpu_bootstrap unpacks to /kaggle/working/src — so cell 0 must have run first.")
print(f"pins from : {_REQ}")

_reqs = [l.split("#", 1)[0].strip() for l in _REQ.read_text(encoding="utf-8").splitlines()]
_reqs = [r for r in _reqs if r]
for _req in _reqs:
    _flags = ["--no-deps"] if _req.startswith("vietocr") else []
    print(f"  pip install {_req} {' '.join(_flags)}", flush=True)
    _p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_flags, _req],
                        capture_output=True, text=True)
    if _p.returncode:
        # ⚠️ The failure is REPORTED with pip's own last words. The previous message guessed
        # "no internet", which was wrong here and sent the diagnosis in the wrong direction.
        print(f"    FAILED rc={_p.returncode}: {(_p.stderr or _p.stdout).strip()[-300:]}")

# `fitz` is pymupdf's import name and `cv2` is opencv's, so the probe cannot be the pin name.
_IMPORTS = ["onnxruntime", "fitz", "vietocr", "cv2", "shapely", "pyclipper", "numpy", "einops"]
_absent = [m for m in _IMPORTS if importlib.util.find_spec(m) is None]
if _absent:
    raise RuntimeError(
        f"{_absent} could not be imported after installing every line of {_REQ.name}. "
        f"Read the FAILED lines above — and if there are none, the kernel has no internet "
        f'(set "enable_internet": true). The MODELS ship in the payload; the packages cannot.')


In [ ]:
# ── MODE ──────────────────────────────────────────────────────────────────────
# ⚠️ **RESOLVED OUT LOUD, AND AN EXPLICIT MODE IS ASSERTED.** `kgpu_bootstrap` sets
# CAFEF_DATA_ROOT to the mounted payload; unset, this is the repo's own raw_data/cafef. Asking
# for "kgpu" where no payload is mounted must RAISE — falling back would parse the repo's own
# files and report a successful Kaggle run that never touched the payload.
import os
import sys
from pathlib import Path

if importlib.util.find_spec("web_scraper") is None:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "web_scraper").is_dir():
            sys.path.insert(0, str(candidate / "src"))
            break
    else:
        raise RuntimeError(f"no src/web_scraper above {here}")

from web_scraper import pdf_ocr_job as job

_payload_root = os.environ.get(job.DATA_ROOT_ENV)
RESOLVED_MODE = MODE if MODE != "auto" else ("kgpu" if _payload_root else "local")
if MODE not in ("auto", "local", "kgpu"):
    raise ValueError(f"MODE must be 'auto', 'local' or 'kgpu', not {MODE!r}")
if RESOLVED_MODE == "kgpu" and not _payload_root:
    raise RuntimeError(
        "MODE='kgpu' but no payload is mounted: kgpu_bootstrap sets CAFEF_DATA_ROOT and it is "
        "unset. Falling back to the repo would report a Kaggle run that read local files."
    )
if RESOLVED_MODE == "local" and _payload_root:
    print(f"WARNING: MODE='local' while a payload IS mounted at {_payload_root} — the run will "
          f"read it, because that is what CAFEF_DATA_ROOT means. Set MODE='kgpu' to say so.")

DATA_ROOT = job.use_data_root()          # $CAFEF_DATA_ROOT on a worker, raw_data/cafef here
MODELS = job.use_models()                # $CAFEF_MODELS_DIR on a worker, src/web_scraper/models here
print(f"MODE      : {RESOLVED_MODE}" + ("  (from MODE='auto')" if MODE == "auto" else ""))
print(f"data root : {DATA_ROOT}")
print(f"models    : det={MODELS.get('det')}\n            vietocr={MODELS.get('vietocr')}")

# ⚠️ **AN ABSENT CHECKPOINT IS A DOWNLOAD, NOT AN ERROR — so it is turned into one here.**
# Without a local `vgg_seq2seq.pth` vietocr fetches ~90 MB from vocr.vn on the first page,
# which fails outright on a kernel with no internet and silently costs a cold start on one
# with it. The payload ships both models; this says so out loud when it did not.
for _which in ("det", "vietocr"):
    if MODELS.get(_which) is None:
        print(f"WARNING: no local {_which} model — the engine will try to DOWNLOAD one.")

import torch
print(f"torch     : {torch.__version__}  cuda={torch.cuda.is_available()}"
      + (f"  {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ""))

# ⚠️ **`get_available_providers()` IS AN ADVERTISEMENT.** On the first Kaggle run it listed
# CUDAExecutionProvider while the session could not create one. `engine_report()` builds the
# detector and reads back what the SESSION holds, which is the only honest answer — and it goes
# into the run's metadata.json so a later reader can tell a GPU-detected run from a CPU one.
ENGINE = job.engine_report()
# ⚠️ **THE FINGERPRINT IS WHAT MAKES TWO RUNS COMPARABLE — OR SAYS THEY ARE NOT.** `torch` is
# on the fingerprinted list and CANNOT be pinned (Kaggle ships its own, and moving this machine
# would invalidate the modelling stack), so the residue is permanent. Two runs whose
# fingerprints differ may be compared on CORRECTNESS and on nothing else.
print(f"stack fp  : {ENGINE.get('stack_fingerprint')}")
_bad = ENGINE.get("pin_violations") or {}
print(f"pins      : {'all honoured' if not _bad else 'VIOLATED — ' + str(_bad)}")
for _k, _v in (ENGINE.get("stack") or {}).items():
    print(f"    {_k:<24} {_v}")
print(f"onnxrt    : {ENGINE.get('onnxruntime')}")
print(f"  advertises  : {ENGINE.get('onnxruntime_advertises')}")
print(f"  DETECTION on: {ENGINE.get('det_providers')}")
print(f"  RECOGNITION : {ENGINE.get('recognizer_device')}")
if "CUDAExecutionProvider" not in (ENGINE.get("det_providers") or []):
    print("WARNING: DB detection is on the CPU (~1.8 s/page against ~0.25 on a GPU). The run "
          "is still correct; it is just paying for the wrong half of the machine.")


In [ ]:
# ── THE INPUT — one object, validated before anything is spent ────────────────
# ⚠️ `JobSpec` is the SAME object the CLI and `kgpu` build. `prepare()` resolves the data root,
# the models, the TEMPLATE and the document list, and raises on any of them — no OCR, no PDF.
SPEC = job.JobSpec(
    exchange=EXCHANGE, symbol=SYMBOL, periods=PERIODS, allow_parent=ALLOW_PARENT,
    template=TEMPLATE, layers=LAYERS, out_root=OUT_ROOT,
    compare_with_disk=COMPARE,
    notes=NOTES or f"MODE={RESOLVED_MODE}",
)

PREPARED = SPEC.prepare()
for _line in PREPARED.describe():
    print(_line)

# ⚠️ **TPL-1 — a resolved template is not a SAFE one.** Two of the seven reconcile anchors are
# bank-shaped, and on `corp` and `insurance` the cash-flow one fuzzy-matches the OPENING balance
# and returns it as the CLOSING one (0.885 / 0.902 against a 0.85 threshold, first hit wins in
# statement order). That is a wrong figure, not a refusal, and no gate below catches it.
if PREPARED.template != "bank":
    print(f"\n⚠️ TPL-1: this is a `{PREPARED.template}` filing and the reconcile anchors are "
          f"bank-shaped.\n   The BALANCE SHEET and INCOME STATEMENT are the trustworthy half; "
          f"a CASH FLOW\n   accepted on this template must be checked against the filing by "
          f"hand before it is quoted.")

for _t in PREPARED.tasks:
    print(f"  {_t.period:<8} {_t.file[:56]:<56} "
          f"{os.path.getsize(_t.path) / 1024 ** 2:>6.1f} MB"
          + ("  CUMULATIVE" if _t.cumulative else ""))


In [ ]:
# ── THE PARSE ─────────────────────────────────────────────────────────────────
# ⚠️ Each document's JSON is written BEFORE the next one starts (§5 rule 20): this stage has
# single documents that cost over half an hour, and a run that keeps its results in memory
# loses every one of them to the first crash. The log below carries three percentages and each
# one names its denominator — only the PAGE fraction predicts time.
folder = job.run(SPEC)
print(folder)


In [ ]:
# ── WHAT CAME OUT ─────────────────────────────────────────────────────────────
# The run folder is the artefact; this reads it back rather than re-using anything in memory,
# so the cell measures what a later reader would actually get.
import json

import pandas as pd

metadata = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
print(f"run     : {metadata['run_id']}  (schema v{metadata['schema_version']})")
print(f"commit  : {metadata['git_commit']}")
print(f"template: {metadata['inputs']['template']}  ({metadata['inputs']['template_how']})")
print(f"gpu     : {metadata['environment']['gpu'].get('name')}"
      f"   detection {metadata['environment']['ocr'].get('det_providers')}")
print(f"took    : {metadata['execution']['runtime']}")

for path in sorted((folder / "documents").glob("*.json")):
    doc = json.loads(path.read_text(encoding="utf-8"))
    print(f"\n{doc['period']}  {doc['document']}  ({doc['seconds'] / 60:.1f} min)")
    for report, entry in (doc.get("compare") or {}).items():
        got = doc["accepted"].get(report)
        head = f"  {report:<18} {entry['verdict']}"
        if got and "identical" in entry:
            head += (f"  {entry['identical']}/{entry['cells_run']} cells identical"
                     f", {len(entry['changed'])} changed"
                     f", layer {entry['run_layer']} vs {entry['disk_layer']} on disk")
        elif got:
            head += f"  [{got['layer']}] {got['items']} items"
        print(head)
        for column, (on_disk, in_run) in list((entry.get("changed") or {}).items())[:5]:
            print(f"      {column}: disk {on_disk:,} -> run {in_run:,}")

summary = pd.DataFrame(metadata["results"])
summary
